<a href="https://colab.research.google.com/github/pcmouadji-dot/deep_learning/blob/main/Contradictory%2C_My_Dear_Watson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import tensorflow as tf
from tensorflow import keras
import keras_nlp
import numpy as np
import pandas as pd

In [5]:
try:
    # detect and init the TPU
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect()
    # instantiate a distribution strategy
    strategy = tf.distribute.TPUStrategy(tpu)
except ValueError:
    print("TPU not activated")
    strategy = tf.distribute.MirroredStrategy() # Works on CPU, single GPU and multiple GPUs in a single VM.

print("replicas:", strategy.num_replicas_in_sync)

TPU not activated
replicas: 1


In [6]:
train_data=pd.read_csv("train.csv")
train_data = train_data.sample(frac=1, random_state=42).reset_index(drop=True)
test_data=pd.read_csv("test.csv")
RESULT_DICT = {
    0 : "entailment",
    1 : "neutral",
    2 : "contradiction"
}

In [7]:
def display_pair_of_sentence(x):
    print( "Premise : " + x['premise'])
    print( "Hypothesis: " + x['hypothesis'])
    print( "Language: " + x['language'])
    print( "Label: " + str(x['label']))
    print()

train_data.head(10).apply(lambda x : display_pair_of_sentence(x), axis=1)

train_data.shape

Premise : Кто? Она спросила его с неожиданным интересом.
Hypothesis: Она спросила, как это сделать, так как с её точки зрения это казалось невозможным.
Language: Russian
Label: 1

Premise : Others are Zao (in Tohoku) and a number of resorts in Joshin-etsu Kogen National Park in the Japan Alps, where there are now splendid facilities thanks to the 1998 Winter Olympic Games in Nagano.
Hypothesis: There are a lot of resorts in the national park.
Language: English
Label: 0

Premise : trying to keep grass alive during a summer on a piece of ground that big was expensive
Hypothesis: There was no cost in keeping the grass alive in the summer time.
Language: English
Label: 2

Premise : so i guess my experience is is just with what we did and and so they didn't really go through the child care route they were able to be home together
Hypothesis: They were able to be home rather than having to worry about getting child care.
Language: English
Label: 0

Premise : The Journal put the point succinc

(12120, 6)

In [8]:
train_size=int(train_data.shape[0]*0.7)
BATCH_SIZE= 16 * strategy.num_replicas_in_sync

In [9]:
def split_labels(x, y):
    return (x[0], x[1]), y


training_dataset = (
    tf.data.Dataset.from_tensor_slices(
        (
            train_data[['premise','hypothesis']].values,
            keras.utils.to_categorical(train_data['label'], num_classes=3)
        )
    )
)

train_dataset = training_dataset.take(train_size)
val_dataset = training_dataset.skip(train_size)


train_preprocessed = (train_dataset
    .map(split_labels, tf.data.AUTOTUNE)
    .shuffle(2048)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE))
val_preprocessed = val_dataset.map(split_labels, tf.data.AUTOTUNE).batch(BATCH_SIZE, drop_remainder=True).cache().prefetch(tf.data.AUTOTUNE)

In [10]:
import tensorflow as tf
print(tf.config.list_physical_devices())
print(strategy.num_replicas_in_sync)

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
1


In [11]:
#from keras.src.optimizers import adam
#with strategy.scope():
 #   preprocessor = keras_nlp.models.XLMRobertaPreprocessor.from_preset(
  #  "xlm_roberta_base_multi", sequence_length=128)
   # model = keras_nlp.models.XLMRobertaClassifier.from_preset(
    #"xlm_roberta_base_multi", preprocessor=preprocessor, num_classes=3)

    #model.compile(optimizer=keras.optimizers.Adam(1e-5*strategy.num_replicas_in_sync),metrics=['accuracy'],loss='categorical_crossentropy')
    #model.summary()

In [12]:
EPOCHS = 6
steps_per_epoch =  train_size // BATCH_SIZE
total_steps = steps_per_epoch * EPOCHS
with strategy.scope():
    preprocessor = keras_nlp.models.BertPreprocessor.from_preset(
        "bert_base_multi", sequence_length=128
    )
    model = keras_nlp.models.BertClassifier.from_preset(
        "bert_base_multi", preprocessor=preprocessor, num_classes=3
    )

    lr = keras.optimizers.schedules.PolynomialDecay(
        initial_learning_rate=2e-5,
        decay_steps=total_steps,
        end_learning_rate=1e-6,
    )

    model.compile(
        optimizer=keras.optimizers.Adam(lr, clipnorm=1.0),
        loss=keras.losses.CategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
    )

    callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2,
                                  restore_best_weights=True),
]

model.fit(
     train_preprocessed,
     validation_data=val_preprocessed,
     epochs=8,
     callbacks=callbacks,
)


100%|██████████| 458/458 [00:00<00:00, 245kB/s]


100%|██████████| 762/762 [00:00<00:00, 856kB/s]


100%|██████████| 972k/972k [00:00<00:00, 87.5MB/s]


100%|██████████| 679M/679M [00:05<00:00, 125MB/s]


Epoch 1/8
530/530 ━━━━━━━━━━━━━━━━━━━━ 363s 635ms/step - accuracy: 0.5565 - loss: 0.9408 - val_accuracy: 0.6468 - val_loss: 0.7956
Epoch 2/8
530/530 ━━━━━━━━━━━━━━━━━━━━ 339s 639ms/step - accuracy: 0.7133 - loss: 0.6937 - val_accuracy: 0.6456 - val_loss: 0.8358
Epoch 3/8
530/530 ━━━━━━━━━━━━━━━━━━━━ 341s 642ms/step - accuracy: 0.8362 - loss: 0.4420 - val_accuracy: 0.6506 - val_loss: 0.9298
Epoch 4/8
530/530 ━━━━━━━━━━━━━━━━━━━━ 340s 641ms/step - accuracy: 0.9086 - loss: 0.2616 - val_accuracy: 0.6578 - val_loss: 1.1749
Epoch 5/8
530/530 ━━━━━━━━━━━━━━━━━━━━ 339s 637ms/step - accuracy: 0.9487 - loss: 0.1550 - val_accuracy: 0.6487 - val_loss: 1.3484
Epoch 6/8
530/530 ━━━━━━━━━━━━━━━━━━━━ 338s 636ms/step - accuracy: 0.9662 - loss: 0.1052 - val_accuracy: 0.6547 - val_loss: 1.4321


In [13]:
predictions=model.predict((test_data['premise'].values, test_data['hypothesis'].values),
    batch_size=32,)

163/163 ━━━━━━━━━━━━━━━━━━━━ 54s 317ms/step


In [14]:
submission = test_data.id.copy().to_frame()
submission["prediction"] = np.argmax(predictions, axis=1)
submission.to_csv("submission.csv",index=False)